In [ ]:
#Part II: Practice the NLP model to classify data stories 


import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Step 1: Load dataset
data = pd.read_csv('data_stories_one_shot.csv')

# Step 2: Map narrative levels to binary labels (0 = Show, 1 = Tell)
data['Target'] = data['Stage'].apply(lambda val: 0 if val == 1 else 1)

# Step 3: Define features, labels, and grouping column
text_data = data['Sentence'].values
target_labels = data['Target'].values
plot_groups = data['Plot_Name'].values

# Step 4: TF-IDF vectorizer with preprocessing
tfidf = TfidfVectorizer(lowercase=True,
                        stop_words='english',
                        token_pattern=r'\b[a-z]{2,}\b')

# Step 5: Define classifiers to test
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'LinearSVM': LinearSVC(class_weight='balanced'),
    'NaiveBayes': MultinomialNB(),
    'RandomForest': RandomForestClassifier(n_estimators=300,
                                           class_weight='balanced',
                                           random_state=42)
}

# Step 6: Stratified 5-Fold Cross Validation
print("=== Stratified 5-Fold Cross Validation ===")
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for model_name, model in models.items():
    pipeline = make_pipeline(tfidf, model)
    accuracy_scores = cross_val_score(pipeline, text_data, target_labels,
                                      cv=stratified_cv, scoring='accuracy')
    print(f"{model_name}: {accuracy_scores.mean():.3f} ± {accuracy_scores.std():.3f}")

# Step 7: Leave-One-Plot-Out Cross Validation
print("\n=== Leave-One-Plot-Out Cross Validation ===")
grouped_cv = GroupKFold(n_splits=len(np.unique(plot_groups)))
for model_name, model in models.items():
    pipeline = make_pipeline(tfidf, model)
    grouped_scores = cross_val_score(pipeline, text_data, target_labels,
                                     cv=grouped_cv, groups=plot_groups,
                                     scoring='accuracy')
    print(f"{model_name}: {grouped_scores.mean():.3f} ± {grouped_scores.std():.3f}")


# In this project, I used TF-IDF to turn sentences into numbers and trained different machine learning models to tell the difference between “Show” and “Tell” sentences. 
# I tested the models using two methods: Stratified 5-Fold and Leave-One-Plot-Out. The results showed that Logistic Regression and Linear SVM worked best, with accuracy 
# around 82% in 5-fold and 80% in LOPO tests. This matches what was shown in the paper’s Figure 6. It means these models are good at finding patterns in how people write stories. In the future, I could try more advanced methods like Sentence-BERT to see if results improve.
# This was done using Python in a Jupyter Notebook, with help from ChatGPT.
#Chatgpt Link: https://chatgpt.com/share/6812e094-c7f8-8008-9c43-4e5a1d759296

=== Stratified 5-Fold Cross Validation ===
LogisticRegression: 0.815 ± 0.075
LinearSVM: 0.800 ± 0.045
NaiveBayes: 0.777 ± 0.066
RandomForest: 0.792 ± 0.062

=== Leave-One-Plot-Out Cross Validation ===
LogisticRegression: 0.804 ± 0.186
LinearSVM: 0.805 ± 0.183
NaiveBayes: 0.711 ± 0.142
RandomForest: 0.623 ± 0.138
